# 심화 미션: 냉동 창고 온도 기록
- 상황: 양품·불량을 찍어주는 사람이 없다. 폐기 기록만 한참 뒤에 붙는다
- 목표: 다섯 날 배운 것을 한 데이터로 전부 돌려본다

### 용어 풀이 - 냉동 창고에서 쓰는 말

| 말 | 뜻 |
|---|---|
| 콜드체인 | 얼거나 차가운 상태를 끊지 않고 이어가는 물류. 한 번 녹으면 되돌릴 수 없다 |
| 제상 | 증발기에 낀 성에를 녹여 떼는 일. 주기적으로 돌리는데 그동안은 온도가 조금 오른다 |
| 복귀 온도 | 창고를 한 바퀴 돌고 설비로 돌아오는 공기의 온도. 내부 온도보다 조금 높다 |
| 압축기 | 냉기를 만드는 설비. 더 식혀야 하면 더 빨리 돈다 |
| 폐기 | 상해서 버린 것. 이 데이터에서 맞혀야 할 정답 |

### 오늘 쓰는 방법의 정식 이름

| 수업에서 쓰는 말 | 정식 이름 |
|---|---|
| 고립시키기 | 아이솔레이션 포레스트 (Isolation Forest) |
| 이웃과 비교하기 | 국소 이상치 인자 (Local Outlier Factor, LOF) |
| 이상 비율 | contamination |
| 이웃 수 | n_neighbors |
| 겹침 | 교집합 (intersection) |
| 지목 중 진짜 | 정밀도 (precision) |
| 전체 폐기 중 잡은 것 | 재현율 (recall) |
| 관리 한계선 | 관리상한 UCL / 관리하한 LCL |
| 관리도로 공정을 지켜보는 일 | 통계적 공정관리 (SPC) |

## Q1. 파일 열고 크기 확인하기

In [5]:
import pandas as pd

# ① 파일을 불러오는 것
df = pd.read_csv("../../data/day05_coldchain.csv")

# ② 행 수와 열 수가 한 덩어리로 담긴 것
print("행", df.shape[0], "/ 열", df.shape[1])

# ③ 참인 자리의 개수를 세는 것
폐기수 = (df["disposal"] == "폐기").sum()
print(f"폐기 {폐기수}건 ({폐기수 / len(df) * 100:.1f}%)")

행 3412 / 열 12
폐기 120건 (3.5%)


## Q2. 시간 순으로 줄 세우기

In [6]:
# ① 글자로 된 시각을 날짜로 바꾸는 것
# ② 표기가 섞여 있어도 읽으라는 뜻
df["recorded_at"] = pd.to_datetime(df["recorded_at"], format="mixed")

# ③ 오름차순으로 정렬돼 있는지 참·거짓으로 알려주는 것
print("파일이 시간 순인가:", df["recorded_at"].is_monotonic_increasing)

# ④ 그 열 기준으로 줄을 세우는 것
df = df.sort_values("recorded_at").reset_index(drop=True)
print("기간:", df["recorded_at"].min(), "~", df["recorded_at"].max())

파일이 시간 순인가: False
기간: 2026-06-01 00:49:00 ~ 2026-09-03 09:19:00


## Q3. 입력 열 고르고 빈칸 채우기

In [8]:
# ① 자료형으로 열을 골라주는 것
숫자열 = df.select_dtypes(include="number").columns.tolist()
print("숫자 열", len(숫자열), "개")

# ② 빈칸이면 참이 되는 것
빈칸 = df[숫자열].isna().sum()
print("빈칸이 있는 열:", 빈칸[빈칸 > 0].to_dict())

# ③ 빈칸을 채우는 것   ④ 가운데 값
X = df[숫자열].fillna(df[숫자열].median())
y = (df["disposal"] == "폐기").astype(int)

숫자 열 7 개
빈칸이 있는 열: {'humidity_pct': 42, 'power_kw': 27}


## Q4. 학습용과 시험용으로 나누기

In [10]:
from sklearn.model_selection import train_test_split

# ① 나눠주는 도구
# ② 시험용으로 떼어둘 비율 (20%)
# ③ 드문 쪽 비율을 양쪽에 똑같이 맞추라는 뜻
학습입력, 시험입력, 학습정답, 시험정답 = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"학습용 {len(학습입력)}건 (폐기 {학습정답.sum()}건)")
print(f"시험용 {len(시험입력)}건 (폐기 {시험정답.sum()}건)")

학습용 2729건 (폐기 96건)
시험용 683건 (폐기 24건)


## Q5. 아무것도 안 하는 기준 모델

In [14]:
# ① 시험용 정답이 정상인 자리만 참 - 정상을 무엇으로 적어뒀나
# ② 참인 비율. 참·거짓에 쓰면 비율이 나온다
기준정확도 = (시험정답 == 0).mean()
print(f"전부 정상이라 답하면 정확도: {기준정확도 * 100:.2f}%")
print("잡은 폐기: 0건")

전부 정상이라 답하면 정확도: 96.49%
잡은 폐기: 0건


## Q6. 손대지 않은 모델 한 번

In [16]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# ① 값을 같은 자로 바꿔주는 것
# ② 셋째 날에 쓴 그 분류 모델
모델 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

# ③ 배우게 하는 것 - 입력과 정답을 함께 준다
모델.fit(학습입력, 학습정답)

# ④ 답을 받는 것 - 정답은 주지 않는다
예측 = 모델.predict(시험입력)

# ⑤ 맞힌 비율
print(f"정확도: {(예측 == 시험정답).mean() * 100:.2f}%")

정확도: 97.80%


## Q7. 네 칸으로 쪼개 보고 지표 구하기